# Day 2

Today, we will start using nf-core pipelines to find differentially abundant genes in our dataset. 
We are using data from the following paper: https://www.nature.com/articles/s41593-023-01350-3#Sec10

## Task 1
Please take some time to read through the paper and understand their approach, hypotheses and goals.

## Task 2
What was the objective of the study?

They hypothesized that there are regulatory pathways that are differentially expressed in mice in 

The objective of the study was to find genes that regulate behaviours manifested by opioid withdrawal that could then be e.g. inhibited to facilitate the transition from opioid-based analgesics (i.e. pain killers) to non-opioid analgesics in patients with chronic pain. This is done because the development of physical dependence and addiction disorders due to misuse of opioids like Oxycodone is a major problem in pain therapeutics.



## Task 3
What do the conditions mean?

**oxy:** oxycodone (opioid)


**sal:** saline (control; non-opioid)

## Task 4 
What do the genotypes mean?

**SNI:**
Prolonged spared nerve injury (long-lasting neuropathic pain)


**Sham:**
No SNI (no pain)



## Task 5
Imagine you are the bioinformatician in the group who conducted this study. They hand you the raw files and ask you to analyze them.

What would you do?

Which groups would you compare to each other?

Please also mention which outcome you would expect to see from each comparison.

**What I would do:**
1. Quality control of RNAseq data
    - 
2. Normalization of RNAseq data
    - batch correction
    - etc.
3. Differential expression analysis 
    - to observe how genes are expressed in one group compared to another
    - to identify potential regulators of opioid withdrawal

4. Treatment of the withdrawal
    - once genes associated with the withdrawal of opioids have been identified, I would repeat the process of differential expression analysis to find out if proposed treatments e.g. RBC1HI could attenuate symptoms of withdrawal

**Which groups I would compare:**
I would compare both controls against each other (**SNI-sal vs. sham-sal**) because I expect them to both not develop opioid dependence or addiction symptoms and thus no differentially expressed genes associated with dependence or addiction symptoms. If this is the case then further analysis can be done.



With the comparison of **sham-oxy and sham-sal** I would expect to see differentially expressed genes associated with opioid dependence and addiction symptoms in mice without any chronic pain.

With the comparison of **SNI-oxy and SNI-sal** I could then find differences between the expression profiles of mice which all have chronic pain but did not all receive opioids as analgesics.
The expected outcome here would be differential expression of genes associated with opioid dependence and addiction symptoms.


Then I would compare **SNI-oxy and sham-sal** in a differential expression analysis to get the differentially expressed genes. These are then likely the genes associated with dependence and addiction symptoms in opioid misuse mixed together with genes expressed in SNI-oxy due to neuropathic pain that are not expressed in sham-sal. This means genes associated with dependence and addiction symptoms are not isolated here (mix of oxy vs. sal and SNI vs. sham). Through the previous comparisons we can rule out which genes' expression solely depends on the pain itself and not the opioid dependency and addiction. 



## Task 6
Your group gave you a very suboptimal excel sheet (conditions_runs_oxy_project.xlsx) to get the information you need for each run they uploaded to the SRA.<br>
So, instead of directly diving into downloading the data and starting the analysis, you first need to sort the lazy table.<br>
Use Python and Pandas to get the table into a more sensible order.<br>
Then, perform some overview analysis and plot the results
1. How many samples do you have per condition?
2. How many samples do you have per genotype?
3. How often do you have each condition per genotype?

In [11]:
import pandas as pd
import numpy as np

df = pd.read_excel("conditions_runs_oxy_project.xlsx")


print("Table before sorting:")
print(df)

# sorting the table for genotype and condition => Controls first
df = df.sort_values(by=["condition: Sal", "Genotype: Sham"])


print("Table after sorting:")
print(df)


Table before sorting:
   Patient          Run RNA-seq  DNA-seq condition: Sal Condition: Oxy  \
0        ?  SRR23195505       x      NaN              x            NaN   
1        ?  SRR23195506       x      NaN            NaN              x   
2        ?  SRR23195507       x      NaN              x            NaN   
3        ?  SRR23195508       x      NaN            NaN              x   
4        ?  SRR23195509       x      NaN            NaN              x   
5        ?  SRR23195510       x      NaN              x            NaN   
6        ?  SRR23195511       x      NaN            NaN              x   
7        ?  SRR23195512       x      NaN              x            NaN   
8        ?  SRR23195513       x      NaN              x            NaN   
9        ?  SRR23195514       x      NaN            NaN              x   
10       ?  SRR23195515       x      NaN              x            NaN   
11       ?  SRR23195516       x      NaN            NaN              x   
12       ?  SRR2

In [17]:
# Task 6.1: How many samples do you have per condition?

num_sal = np.count_nonzero(pd.isnull(df["condition: Sal"]))
num_oxy = np.count_nonzero(pd.isnull(df["Condition: Oxy"]))

print("Number of samples per condition:")
print(f"     Sal: {num_sal}")
print(f"     Oxy: {num_oxy}")





Number of samples per condition:
     Sal: 8
     Oxy: 8


In [18]:
# Task 6.2: How many samples do you have per genotype?
num_sni = np.count_nonzero(pd.isnull(df["Genotype: SNI"]))
num_sham = np.count_nonzero(pd.isnull(df["Genotype: Sham"]))

print("Number of samples per genotype:")
print(f"     Sham: {num_sham}")
print(f"     SNI: {num_sni}")


Number of samples per genotype:
     Sham: 8
     SNI: 8


In [24]:
# Task 6.3: How often do you have each condition per genotype?


dict = {}
for geno in ["SNI", "Sham"]:
    sal_mask = df[f"Genotype: {geno}"].notna() & df["condition: Sal"].notna()
    num_sal = sal_mask.sum()
    oxy_mask = df[f"Genotype: {geno}"].notna() & df["Condition: Oxy"].notna()
    num_oxy = oxy_mask.sum()
    dict.update({f"{geno}, Sal": num_sal})
    dict.update({f"{geno}, Oxy": num_oxy})

for k, v in dict.items():
    print(f"{k}: {v}")


SNI, Sal: 4
SNI, Oxy: 4
Sham, Sal: 4
Sham, Oxy: 4


## Task 7
They were so kind to also provide you with the information of the number of bases per run, so that you can know how much space the data will take on your Cluster.<br>
Add a new column to your fancy table with this information (base_counts.csv) and sort your dataframe according to this information and the condition.

Then select the 2 smallest runs from your dataset and download them from SRA (maybe an nf-core pipeline can help here?...)

In [33]:
base_counts = pd.read_csv("base_counts.csv")

merged_df = pd.merge(df, base_counts, on="Run")



# sort runs by size (ascending)
merged_df = merged_df.sort_values(by="Bases", ascending=True)

# get two smallest
print("Two smallest runs based on base count:")
print(merged_df.iloc[0]["Run"])
print(merged_df.iloc[1]["Run"])
print(merged_df)

# print to csv
merged_df.iloc[:2]["Run"].to_csv("runs.csv", index=False, header=False)

Two smallest runs based on base count:
SRR23195516
SRR23195511
   Patient          Run RNA-seq  DNA-seq condition: Sal Condition: Oxy  \
14       ?  SRR23195516       x      NaN            NaN              x   
9        ?  SRR23195511       x      NaN            NaN              x   
15       ?  SRR23195517       x      NaN            NaN              x   
4        ?  SRR23195505       x      NaN              x            NaN   
12       ?  SRR23195508       x      NaN            NaN              x   
11       ?  SRR23195519       x      NaN            NaN              x   
13       ?  SRR23195509       x      NaN            NaN              x   
10       ?  SRR23195514       x      NaN            NaN              x   
5        ?  SRR23195510       x      NaN              x            NaN   
1        ?  SRR23195512       x      NaN              x            NaN   
3        ?  SRR23195520       x      NaN              x            NaN   
8        ?  SRR23195506       x      NaN         

In [ ]:
# nf-core pipeline for getting runs from the SRA
!nextflow run nf-core/fetchngs -r 1.13.0 -profile docker --outdir sra_download --input runs.csv

## Task 8
While your files are downloading, get back to the paper and explain how you would try to reproduce the analysis.<br>
When you are done with this shout, so we can discuss the different ideas.

I would try to follow the Methods section of the paper as close as possible and for any gaps in the explanation that occur I would research the most widely used tools and methods that similar papers use. 

For this paper, the bioinformatic analysis steps described in the Methods section are roughly:
1. Perform read alignment, read counting and differential analysis using HISAT2, HT-Seq33 and the DESeq2 R package, respectively
2. Perform differential analysis for dissecting oxycodone withdrawal vs. saline treatment effects in SNI and sham mice using a 2 × 2 factorial design with the following formula: log(exp) ∼ SNI + oxycodone + SNI: oxycodone *TODO*
3. Define P-value cutoff for differential lists as < 0.05 and log2(fold change) <−0.5 or >0.5
4. Choose a cutoff of −log10(P value) > 1.3 for pathway analysis
5. Rank–rank hypergeometric overlap RNA-seq analysis as described in the respective section